# Эмбеддинги изображений

In [10]:
from PIL import Image
from pathlib import Path
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
from backbone_model import BackboneModel

In [5]:
model = BackboneModel()

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [6]:
model_clf = BackboneModel(task='classification')

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [9]:
device

'cuda'

In [10]:
val_data = pd.read_parquet('embeddings&markup/image_markup.pq')

In [12]:
image_paths = [str(Path('data') / n) for n in val_data['name'].values]

**Строим эмбеддинги и логиты классов всех изображений из размеченной выборки**

In [14]:
embeddings = []
embeddings_clf = []
for pth in tqdm(image_paths):
    embeddings.append(model.extract_embedding(pth))
    embeddings_clf.append(model_clf.predict_logits(pth))

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 299/299 [01:11<00:00,  4.19it/s]


In [15]:
emb_df = pd.DataFrame()
emb_df['image_path'] = image_paths
emb_df['embedding'] = embeddings
emb_df['embedding_clf'] = embeddings_clf

In [16]:
emb_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 299 entries, 0 to 298
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   image_path     299 non-null    object
 1   embedding      299 non-null    object
 2   embedding_clf  299 non-null    object
dtypes: object(3)
memory usage: 7.1+ KB


In [17]:
emb_df.head(3)

,image_path,embedding,embedding_clf
0,/cv-sandbox/project/data/IMG_20221201_131637.jpg,"[-2.8007195, 0.1795005, 2.0634825, 0.31013796,...","[-1.198553, -0.61378217, -0.49053022, -1.18711..."
1,/cv-sandbox/project/data/IMG_20220422_185636.jpg,"[-0.39972767, 0.23981501, 1.5460123, -1.038316...","[-3.2148683, -1.8561819, -3.4543087, -2.958324..."
2,/cv-sandbox/project/data/P7081020.JPG,"[0.8322267, 0.17672215, -0.3619432, 0.58000106...","[-1.4701422, 0.23119043, -1.671558, 1.1484444,..."


## Цветовые вектора

In [1]:
def get_pixel_coords(rgb, bins):
    coords = np.zeros(3)
    N = len(bins)
    for k, color in enumerate(rgb):
        for l, b in enumerate(bins):
            if color < b:
                coords[k] = l
                break
    return coords

In [5]:
n_bins=4

In [20]:
def make_color_vectors(image_paths, n_bins=6):
    color_vectors = []
    bins = np.linspace(0, 256, n_bins + 1).astype(np.int32)[1:]
    for pth in tqdm(image_paths):
        image = Image.open(pth).convert('RGB')
        N = image.size[0] * image.size[1]
        color_vector = np.zeros(np.power(n_bins, 3))
        for i in range(image.size[0]):
            for j in range(image.size[1]):
                rgb = image.getpixel((i, j))
                coords = get_pixel_coords(rgb, bins)
                ind = int(n_bins * n_bins * coords[0] + n_bins * coords[1] + coords[2])
                color_vector[ind] += 1
        color_vectors.append((pth, color_vector / N))
    return color_vectors

In [ ]:
color_vectors = make_color_vectors(emb_df['image_path'].values)

In [ ]:
emb_df['color_vectors'] = color_vectors

---

In [49]:
emb_df.to_parquet('image_embeddings.pq')